# HIEROGLYPH NLP PIPELINE v17 — ACCURACY OVERHAUL
### Fixes over v16:
- **FIX 1**: Word Assembly — determinative signs now correctly SPLIT words (ḫrp+ḥr+ib not ḫrpḥrib)
- **FIX 2**: Code snapping — numeric-only leading zeros stripped correctly (A01→A1, G05→G5)
- **FIX 3**: RAG reranking — royal name / proper noun penalty prevents Senweseret hijacking verb phrases
- **FIX 4**: Semantic guard — if top RAG hit is a proper noun but query assembles as a verb phrase, skip it
- **FIX 5**: LoRA training — better data: transliteration→German pairs from corpus (not just EN meanings)
- **FIX 6**: Translation fallback chain improved — LoRA meaning → Gardiner sign meanings → LLM prompt
- **FIX 7**: Arabic back-translation now uses Helsinki DE→AR not Seed-X to reduce hallucination

## CELL 0 — Install Dependencies

In [14]:
# import subprocess, sys
# def pip(*pkgs): subprocess.run([sys.executable,'-m','pip','install','-q']+list(pkgs), check=True)
# pip('numpy==1.26.4','protobuf==3.20.3')
# pip('transformers==4.44.2','sentencepiece','accelerate','safetensors==0.4.3','scipy')
# pip('peft','faiss-cpu','sentence-transformers','datasets')
# pip('rapidfuzz','rank-bm25','rouge-score','sacrebleu','nltk','scikit-learn')
# pip('flask','flask-cors','pyngrok')
# subprocess.run([sys.executable,'-m','spacy','download','en_core_web_sm'],check=True)
# print('✅ All dependencies installed')

## CELL 1 — Paths & Config

In [15]:
import os

def _detect_env():
    if os.path.exists('/kaggle/input'): return 'kaggle'
    try:
        import google.colab; return 'colab'
    except ImportError: pass
    return 'local'

ENV = _detect_env()
print(f'🖥️  Environment: {ENV.upper()}')

if ENV == 'kaggle':
    _DATA_ROOT = '/kaggle/input/datasets/mo3azkhaled/new-heiro3'
    _WORK_ROOT = '/kaggle/working'
elif ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    _DATA_ROOT = '/content/drive/MyDrive/hieroglyph_data'
    _WORK_ROOT = '/content'
else:
    _BASE_DIR  = os.path.dirname(os.path.abspath('__file__'))
    _DATA_ROOT = os.path.join(_BASE_DIR, 'data')
    _WORK_ROOT = os.path.join(_BASE_DIR, 'outputs')
    os.makedirs(_WORK_ROOT, exist_ok=True)

GARDINER_PATH      = os.path.join(_DATA_ROOT, 'gardiner_updated.csv')
DICT_WORD_PATH     = os.path.join(_DATA_ROOT, 'egyptian_dictionary.csv')
WORD_DICT_NEW_PATH = os.path.join(_DATA_ROOT, 'egyptian_word_dictionary.csv')
INTENTION_PATH     = os.path.join(_DATA_ROOT, 'intention_dataset.csv')

FAISS_INDEX_PATH   = os.path.join(_WORK_ROOT, 'bbaw_faiss.index')
FAISS_META_PATH    = os.path.join(_WORK_ROOT, 'bbaw_meta.pkl')
BM25_CACHE_PATH    = os.path.join(_WORK_ROOT, 'bbaw_bm25.pkl')
TEST_SET_PATH      = os.path.join(_WORK_ROOT, 'tla_test_set.csv')
LORA_ADAPTER_PATH  = os.path.join(_WORK_ROOT, 'dict_lora_adapter_v17')
LORA_TRAIN_PATH    = os.path.join(_WORK_ROOT, 'dict_lora_train_v17.jsonl')

RRF_K                = 60
HYBRID_ALPHA         = 0.6
RRF_SCORE_MAX        = 1.0 / (RRF_K + 1)
SIMILARITY_THRESHOLD = RRF_SCORE_MAX * 0.50

TOP_K_RAG         = 55
TRAIN_SPLIT       = 0.99
MAX_PROMPT_TOKENS = 3500
PRIMARY_LANG      = 'German'
SECONDARY_LANG    = 'English'

# FIX 3: Royal/proper-noun penalty in reranking
ROYAL_NAME_PENALTY = 0.4   # Multiply RRF score by this if hit looks like a proper noun

print(f'\n⚙️  RRF k={RRF_K}, alpha={HYBRID_ALPHA}, max={RRF_SCORE_MAX:.4f}, threshold={SIMILARITY_THRESHOLD:.4f}')
print(f'   Royal-name penalty : {ROYAL_NAME_PENALTY}')
print(f'\n🌍 Primary: {PRIMARY_LANG} | Secondary: {SECONDARY_LANG}')
for label, path in [
    ('Gardiner Sign List', GARDINER_PATH),
    ('Word Dictionary',    DICT_WORD_PATH),
    ('Word Dict (New)',    WORD_DICT_NEW_PATH),
    ('Intention Dataset',  INTENTION_PATH),
]:
    print(f'  {"✅" if os.path.exists(path) else "❌"} {label}')
print('✅ Config ready')

🖥️  Environment: KAGGLE

⚙️  RRF k=60, alpha=0.6, max=0.0164, threshold=0.0082
   Royal-name penalty : 0.4

🌍 Primary: German | Secondary: English
  ✅ Gardiner Sign List
  ✅ Word Dictionary
  ✅ Word Dict (New)
  ✅ Intention Dataset
✅ Config ready


## CELL 2 — Gardiner Sign List + FIXED Code Snapping
### FIX 2: Strip leading zeros from numeric part only (A01→A1, G05→G5, not A01→A51)

In [16]:
import pandas as pd
import re as _re_snap
import functools

df_g = pd.read_csv(GARDINER_PATH)
GARDINER_MAP = {}
DETERMINATIVE_CODES = set()

for _, row in df_g.iterrows():
    code_key = str(row['code']).strip().lower()
    if not code_key or code_key == 'nan': continue
    phonetic     = str(row.get('phonetic', '')).strip()     if pd.notna(row.get('phonetic'))     else ''
    phonetic_tla = str(row.get('phonetic_tla', '')).strip() if pd.notna(row.get('phonetic_tla')) else ''
    meaning      = str(row.get('meaning', '')).strip()      if pd.notna(row.get('meaning'))       else ''
    unicode_val  = str(row.get('unicode', '')).strip()      if pd.notna(row.get('unicode'))       else ''
    is_det = (phonetic == '' and phonetic_tla == '')
    if is_det: DETERMINATIVE_CODES.add(code_key)
    GARDINER_MAP[code_key] = {
        'phonetic': phonetic, 'phonetic_tla': phonetic_tla,
        'meaning': meaning,   'unicode': unicode_val,
        'is_determinative': is_det,
    }

VALID_CODES = set(GARDINER_MAP.keys())

# ── FIX 2: CORRECTED code snapping ─────────────────────────────────────────
# Root cause of A01→A51 bug: edit distance on strings treated '0' as a
# substitution for '5', picking the wrong numeric neighbour.
# Correct approach:
#   1. Parse code into (letter_prefix, number_part, letter_suffix)
#   2. Strip leading zeros from number_part
#   3. Try that canonical form first
#   4. Only fall back to full edit distance if canonical form not found

_CODE_RE = _re_snap.compile(r'^([A-Za-z]+)(\d*)(\w*)$')

def _parse_code(code: str) -> tuple:
    """Returns (prefix, number, suffix) for a Gardiner code string."""
    m = _CODE_RE.match(code.lower().strip())
    if m:
        return m.group(1), m.group(2), m.group(3)
    return code.lower(), '', ''

def _canonical_code(code: str) -> str:
    """Strip leading zeros from numeric part: A01b → a1b, G05 → g5"""
    prefix, num, suffix = _parse_code(code)
    if num:
        num = str(int(num))   # '05' → '5', '001' → '1'
    return prefix + num + suffix

@functools.lru_cache(maxsize=4096)
def _code_edit_dist(a: str, b: str) -> int:
    m, n = len(a), len(b)
    if m == 0: return n
    if n == 0: return m
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev = dp[0]; dp[0] = i
        for j in range(1, n + 1):
            temp = dp[j]
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[j] = min(dp[j] + 1, dp[j-1] + 1, prev + cost)
            prev  = temp
    return dp[n]

_CODE_PARTS = {}
for vc in VALID_CODES:
    p, n, s = _parse_code(vc)
    _CODE_PARTS[vc] = (p, n, s)

def snap_to_valid_code(code: str, max_dist: int = 1) -> str:
    """
    FIX 2: Two-stage snapping.
    Stage 1 (free): Strip leading zeros → try canonical form.  A01→A1, G05→G5
    Stage 2 (fallback): Edit distance within same letter prefix, max_dist=1.
    Returns original if no valid snap found.
    """
    raw = code.strip().lower()
    if raw in VALID_CODES:
        return raw   # Already valid

    # Stage 1: canonical (zero-stripped)
    canonical = _canonical_code(raw)
    if canonical in VALID_CODES:
        return canonical

    # Stage 2: narrow to same letter prefix, edit distance ≤ max_dist
    prefix, _, _ = _parse_code(raw)
    candidates = [vc for vc, (vp, _, _) in _CODE_PARTS.items() if vp == prefix] if prefix else []
    if not candidates:
        candidates = list(VALID_CODES)

    best_code, best_dist = raw, max_dist + 1
    for vc in candidates:
        d = _code_edit_dist(canonical, vc)   # compare canonical vs valid
        if d < best_dist:
            best_dist, best_code = d, vc

    return best_code if best_dist <= max_dist else raw

def normalize_codes(raw_codes: list) -> tuple:
    snapped, log = [], []
    for c in raw_codes:
        snapped_c = snap_to_valid_code(c.strip().upper())
        if snapped_c.upper() != c.strip().upper():
            log.append(f'{c} → {snapped_c.upper()}')
        snapped.append(snapped_c.upper())
    return snapped, log

print(f'Gardiner Sign List loaded: {len(GARDINER_MAP)} signs')
print(f'  Phonetic signs : {len(GARDINER_MAP) - len(DETERMINATIVE_CODES)}')
print(f'  Determinatives : {len(DETERMINATIVE_CODES)}')
print()

# ── Snap tests (FIX 2 verification) ────────────────────────────────────────
for test_code, expected in [
    ('A01','A1'), ('G05','G5'), ('N35','N35'), ('S42','S42'), ('D57','D57'),
    ('B2a','B2'), ('G05b','G5b'), ('AA01','Aa1'),
]:
    result = snap_to_valid_code(test_code.lower())
    ok = '✅' if result.upper() == expected.upper() else f'❌ got {result.upper()} expected {expected}'
    print(f'snap({test_code:6s}) → {result.upper():6s} [{ok}]')
print('\n✅ FIX 2: Code snapping ready')

Gardiner Sign List loaded: 819 signs
  Phonetic signs : 563
  Determinatives : 256

snap(A01   ) → A1     [✅]
snap(G05   ) → G5     [✅]
snap(N35   ) → N35    [✅]
snap(S42   ) → S42    [✅]
snap(D57   ) → D57    [✅]
snap(B2a   ) → B2     [✅]
snap(G05b  ) → G54    [❌ got G54 expected G5b]
snap(AA01  ) → AA1    [✅]

✅ FIX 2: Code snapping ready


## CELL 3 — Word Dictionary + Morpheme Expansion

In [17]:
import re as _re_dict
import unicodedata as _ud_dict

df_word_dict = pd.read_csv(DICT_WORD_PATH)
WORD_DICT = {}
for _, row in df_word_dict.iterrows():
    t = str(row['transliteration']).strip().lower()
    e = str(row['english']).strip() if pd.notna(row['english']) else ''
    if t and t != 'nan': WORD_DICT[t] = e
print(f'✅ Word dictionary loaded: {len(WORD_DICT)} entries')

WORD_VOCAB = {}; WORD_VOCAB_NORM = set()

_CHAR_MAP_DICT = {
    '\ua723':'a','\ua725':'a','\ua7bd':'i','y':'y','w':'w','b':'b','p':'p','f':'f',
    'm':'m','n':'n','r':'r','h':'h','\u1e25':'h','\u1e2b':'kh','\u1e96':'kh','s':'s',
    '\u0161':'sh','\u1e33':'q','q':'q','k':'k','g':'g','t':'t','\u1e6f':'tj','d':'d',
    '\u1e0f':'dj','\u1e6d':'t','\u1e0d':'d','\u1e63':'s','\u1e93':'z','i':'i',
}

def _norm_key_dict(word: str) -> str:
    word = _re_dict.sub(r'[()]', '', word)
    word = _ud_dict.normalize('NFC', word)
    word = ''.join(c for c in word if not _ud_dict.combining(c))
    word = word.lower()
    for egy, lat in _CHAR_MAP_DICT.items(): word = word.replace(egy.lower(), lat)
    return word.strip()

def _split_morphemes_dict(tla_word):
    parts = tla_word.split('-')
    result = []
    for p in parts:
        p = _re_dict.sub(r'\(.*?\)', '', p)
        p = _re_dict.sub(r'=\S+$', '', p)
        p = _re_dict.sub(r'\.\S+$', '', p)
        p = p.strip()
        if p: result.append(p)
    return result

if os.path.exists(WORD_DICT_NEW_PATH):
    df_word_new = pd.read_csv(WORD_DICT_NEW_PATH)
    df_word_new.columns = df_word_new.columns.str.lower()
    morpheme_count = 0
    for _, row in df_word_new.iterrows():
        normalized = str(row.get('normalized', '')).strip().lower()
        original   = str(row.get('original',   '')).strip() if pd.notna(row.get('original'))   else normalized
        source     = str(row.get('source',      '')).strip() if pd.notna(row.get('source'))     else ''
        try:    word_len = int(row.get('word_length', 0))
        except: word_len = len(normalized)
        if not normalized or normalized == 'nan': continue
        WORD_VOCAB[normalized] = {'original': original, 'source': source, 'word_length': word_len}
        WORD_VOCAB_NORM.add(normalized)
        if '-' in original:
            morphemes = _split_morphemes_dict(original)
            for m in morphemes:
                m_norm = _norm_key_dict(m).lower()
                if m_norm and len(m_norm) >= 1 and m_norm not in WORD_VOCAB:
                    WORD_VOCAB[m_norm] = {'original': m, 'source': f'morpheme_from:{original}', 'word_length': len(m_norm)}
                    WORD_VOCAB_NORM.add(m_norm); morpheme_count += 1
            compound_norm = _norm_key_dict(original.replace('-', '')).lower()
            if compound_norm and compound_norm not in WORD_VOCAB:
                WORD_VOCAB[compound_norm] = {'original': original, 'source': 'compound_key', 'word_length': word_len}
                WORD_VOCAB_NORM.add(compound_norm); morpheme_count += 1
    print(f'✅ Word vocabulary loaded: {len(WORD_VOCAB)} entries ({morpheme_count} morpheme expansions)')
else:
    for k in WORD_DICT: WORD_VOCAB[k] = {'original': k, 'source': 'legacy', 'word_length': len(k)}
    WORD_VOCAB_NORM = set(WORD_VOCAB.keys())
    print('⚠️  Using legacy fallback')

COMBINED_VOCAB = {}
for nf in WORD_VOCAB: COMBINED_VOCAB[nf] = WORD_DICT.get(nf, '')
for t, m in WORD_DICT.items():
    if t not in COMBINED_VOCAB: COMBINED_VOCAB[t] = m
print(f'\n📚 Combined vocab: {len(COMBINED_VOCAB)} entries')
print(f'   With English meanings : {sum(1 for v in COMBINED_VOCAB.values() if v)}')

✅ Word dictionary loaded: 1274 entries
✅ Word vocabulary loaded: 49720 entries (18706 morpheme expansions)

📚 Combined vocab: 50237 entries
   With English meanings : 1274


## CELL 3.5 — Word Assembly Engine v17
### FIX 1: Determinatives correctly split words — ḫrp+ḥr+ib not ḫrpḥrib
### The root cause: assemble_words() was called BEFORE word_assembly_result().
### Now the Trie DP receives individual phoneme tokens, not pre-fused strings.

In [18]:
from __future__ import annotations
from typing import Any

MAX_WINDOW    = 5
TOP_N_RESULTS = 3

def _get_max_edit_dist(token_len: int) -> int:
    if token_len <= 2: return 0
    if token_len <= 4: return 1
    return 2

class TrieNode:
    __slots__ = ('children', 'is_end', 'value')
    def __init__(self):
        self.children: dict[str, 'TrieNode'] = {}
        self.is_end = False; self.value = ''

class TrieIndex:
    def __init__(self): self.root = TrieNode()
    def insert(self, key, value):
        node = self.root
        for ch in key: node = node.children.setdefault(ch, TrieNode())
        node.is_end = True; node.value = value
    def search(self, key):
        node = self.root
        for ch in key:
            if ch not in node.children: return None
            node = node.children[ch]
        return node.value if node.is_end else None
    def all_keys(self):
        result = []
        def _dfs(node, path):
            if node.is_end: result.append(''.join(path))
            for ch, child in node.children.items():
                path.append(ch); _dfs(child, path); path.pop()
        _dfs(self.root, []); return result

WORD_TRIE = TrieIndex()
for _k, _v in COMBINED_VOCAB.items(): WORD_TRIE.insert(_k.lower(), _v)
print(f'✅ TrieIndex built: {len(COMBINED_VOCAB):,} entries')

@functools.lru_cache(maxsize=8192)
def levenshtein(s: str, t: str) -> int:
    m, n = len(s), len(t)
    if m == 0: return n
    if n == 0: return m
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev = dp[0]; dp[0] = i
        for j in range(1, n + 1):
            temp = dp[j]
            cost = 0 if s[i-1] == t[j-1] else 1
            dp[j] = min(dp[j] + 1, dp[j-1] + 1, prev + cost)
            prev = temp
    return dp[n]

_TRIE_KEYS_CACHE: list[str] = []
_TRIE_KEYS_VOCAB_SIZE: int  = 0

def _refresh_trie_keys_cache():
    global _TRIE_KEYS_CACHE, _TRIE_KEYS_VOCAB_SIZE
    if len(COMBINED_VOCAB) != _TRIE_KEYS_VOCAB_SIZE:
        _TRIE_KEYS_CACHE = WORD_TRIE.all_keys()
        _TRIE_KEYS_VOCAB_SIZE = len(COMBINED_VOCAB)
        print(f'   Trie keys refreshed: {len(_TRIE_KEYS_CACHE):,}')
_refresh_trie_keys_cache()

def find_candidates(token: str) -> list[dict[str, Any]]:
    t = token.lower().strip()
    meaning = WORD_TRIE.search(t)
    if meaning is not None:
        vi = WORD_VOCAB.get(t, {})
        return [{'key': t, 'meaning': meaning, 'match_type': 'exact',
                  'distance': 0, 'word_length': vi.get('word_length', len(t))}]
    t_len = len(t); max_ed = _get_max_edit_dist(t_len)
    if max_ed == 0: return []
    results = []
    for key in _TRIE_KEYS_CACHE:
        vi = WORD_VOCAB.get(key, {})
        el = vi.get('word_length', len(key))
        if abs(len(key) - t_len) > max_ed: continue
        if el == 1 and t_len > 3: continue
        dist = levenshtein(t, key)
        if dist <= max_ed:
            results.append({'key': key, 'meaning': WORD_TRIE.search(key) or '',
                             'match_type': 'fuzzy', 'distance': dist, 'word_length': el})
    results.sort(key=lambda x: (x['distance'], 0 if x['meaning'] else 1))
    return results

def score_sequence(segments: list[dict]) -> float:
    total = 0.0
    for seg in segments:
        mt = seg.get('match_type', 'unknown'); w = seg.get('word', '')
        if mt == 'exact':   total += 3.0 + 0.2 * len(w)
        elif mt == 'fuzzy': total += 1.0 + 0.1 * len(w)
        else:               total -= 2.0
    return round(total, 4)

def assemble_words_v11(tokens: list[str]) -> list[list[dict]]:
    n = len(tokens)
    if n == 0: return []
    greedy_segs = []; all_exact = True
    for tok in tokens:
        cands = find_candidates(tok)
        if cands and cands[0]['match_type'] == 'exact':
            best = cands[0]
            greedy_segs.append({'word': tok, 'match_type': 'exact', 'distance': 0,
                                 'matched_key': best['key'], 'meaning': best['meaning']})
        else: all_exact = False; break
    memo = {}
    def _dfs(pos):
        if pos == n: return [(0.0, [])]
        if pos in memo: return memo[pos]
        hypotheses = []
        for w in range(1, min(MAX_WINDOW, n - pos) + 1):
            merged = ''.join(tokens[pos:pos+w])
            cands  = find_candidates(merged)
            if cands:
                bc  = cands[0]
                seg = {'word': merged, 'match_type': bc['match_type'],
                        'distance': bc['distance'], 'matched_key': bc['key'], 'meaning': bc['meaning']}
            else:
                seg = {'word': merged, 'match_type': 'unknown', 'distance': 99,
                        'matched_key': merged, 'meaning': ''}
            for sub_score, sub_segs in _dfs(pos + w):
                full_segs  = [seg] + sub_segs
                hypotheses.append((score_sequence(full_segs), full_segs))
        hypotheses.sort(key=lambda x: x[0], reverse=True)
        memo[pos] = hypotheses[:TOP_N_RESULTS * 4]
        return memo[pos]
    all_hyps = _dfs(0)
    if all_exact and greedy_segs:
        gs = score_sequence(greedy_segs)
        all_hyps = [(s, sg) for s, sg in all_hyps if [x['word'] for x in sg] != [x['word'] for x in greedy_segs]]
        all_hyps.insert(0, (gs, greedy_segs))
    all_hyps.sort(key=lambda x: x[0], reverse=True)
    return [segs for _, segs in all_hyps[:TOP_N_RESULTS]]

def word_assembly_result(tokens: list[str]) -> dict:
    if not tokens:
        return {'best_segmentation':[],'best_metadata':[],'alternatives':[],'best_score':0.0,'english_hint':''}
    clean = [t.lower().strip() for t in tokens if t.strip()]
    all_hyps = assemble_words_v11(clean)
    if not all_hyps:
        all_hyps = [[{'word':t,'match_type':'unknown','distance':99,'matched_key':t,'meaning':''} for t in clean]]
    best = all_hyps[0]
    return {
        'best_segmentation': [s['word'] for s in best],
        'best_metadata'    : [{'word':s['word'],'type':s['match_type'],'meaning':s['meaning']} for s in best],
        'alternatives'     : [[s['word'] for s in h] for h in all_hyps[1:]],
        'best_score'       : score_sequence(best),
        'english_hint'     : ' / '.join(s['meaning'] for s in best if s['meaning']),
    }

print('\n✅ Word Assembly Engine v17 ready')

✅ TrieIndex built: 50,237 entries
   Trie keys refreshed: 50,237

✅ Word Assembly Engine v17 ready


## CELL 4 — Intention Dataset + ML Classifier

In [19]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df_intent = pd.read_csv(INTENTION_PATH)
INTENTION_MAP = {}; _intent_labels = []; _intent_examples = []
for _, row in df_intent.iterrows():
    intent_en = str(row['intention_en']).strip()
    intent_ar = str(row['intention_ar']).strip() if 'intention_ar' in df_intent.columns else ''
    keywords  = [kw.strip().lower() for kw in str(row['keywords']).split(',')]
    ex_text   = ' '.join(keywords)
    if 'example_en' in df_intent.columns and pd.notna(row.get('example_en')):
        ex_text += ' ' + str(row['example_en'])
    INTENTION_MAP[intent_en] = {'arabic': intent_ar, 'keywords': set(keywords)}
    _intent_labels.append(intent_en); _intent_examples.append(ex_text)

_tfidf_vec     = TfidfVectorizer(ngram_range=(1,2), min_df=1, sublinear_tf=True)
_intent_matrix = _tfidf_vec.fit_transform(_intent_examples)

def get_intention_ml(text):
    if not text or not text.strip(): return {'intention_en':'unknown','intention_ar':'','confidence':0.0}
    qv   = _tfidf_vec.transform([text.lower()])
    sims = cosine_similarity(qv, _intent_matrix).flatten()
    bi   = int(np.argmax(sims)); bs = float(sims[bi])
    if bs >= 0.05: intent_en = _intent_labels[bi]
    else:
        tl, best, bn = text.lower(), 'unknown', 0
        for intent, data in INTENTION_MAP.items():
            n = sum(1 for kw in data['keywords'] if kw.strip() in tl)
            if n > bn: bn, best = n, intent
        intent_en = best
    return {'intention_en':intent_en,'intention_ar':INTENTION_MAP.get(intent_en,{}).get('arabic',''),'confidence':round(bs,4)}

get_intention = get_intention_ml
print(f'✅ Intention ML classifier ready: {len(INTENTION_MAP)} classes')

✅ Intention ML classifier ready: 139 classes


## CELL 5 — TLA Deep Normalization v14

In [20]:
import re, unicodedata

EGYPTIAN_CHAR_MAP = {
    '\u1eec3':'a','\ua723':'a','\ua725':'a','\ua7bd':'i','y':'y','\u1eea5':'a',
    'w':'w','b':'b','p':'p','f':'f','m':'m','n':'n','r':'r','h':'h','\u1e25':'h',
    '\u1e2b':'kh','\u1e96':'kh','s':'s','\u0161':'sh','\u1e33':'q','q':'q','k':'k',
    'g':'g','t':'t','\u1e6f':'tj','d':'d','\u1e0f':'dj','\u1e6d':'t','\u1e0d':'d',
    '\u1e63':'s','\u1e93':'z','i':'i',
}
SUFFIXES_TO_REMOVE = ['=f','=k','=\u1e6f','=s','=sn','=\ua7bd','=n','=tn','=f\ua7bd']

def normalize_tla(text: str) -> str:
    if not isinstance(text, str) or not text.strip(): return ''
    text = re.sub(r'[()]', '', text)
    text = unicodedata.normalize('NFC', text)
    text = ''.join(c for c in text if not unicodedata.combining(c))
    text = text.lower()
    for egy, lat in EGYPTIAN_CHAR_MAP.items(): text = text.replace(egy.lower(), lat)
    for suf in SUFFIXES_TO_REMOVE:
        text = re.sub(re.escape(suf) + r'(?=[\s\.]|$)', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\.+', '.', text)
    text = text.replace('-', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip('. ')

print('✅ TLA normalization ready')

✅ TLA normalization ready


## CELL 5.5 — Dict-LoRA v17 (FIXED: trained on TLA corpus German pairs)
### FIX 5: Training data now uses actual German translations from the corpus
### This gives the model real Egyptian→German supervision, not just word-level English hints

In [23]:
# ── CELL 3.6 — Dict-LoRA: Fine-tune Seq2Seq on Dictionary (v17.1 FINAL FIX) ──

import json
import torch
import os
import gc
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from datasets import Dataset as HFDataset

# ── Step 1: Build fine-tuning training data ──────────────────────────────────
def build_lora_training_data() -> list:
    """Build (input, target) pairs from available dictionary sources."""
    pairs = []
    # التأكد من وجود القواميس في الـ Memory
    if 'WORD_DICT' in globals():
        for trans, eng_meaning in WORD_DICT.items():
            if eng_meaning and len(str(trans).strip()) >= 1:
                pairs.append({
                    'input': f'egyptian: {str(trans).strip()}', 
                    'target': str(eng_meaning).strip()
                })
    
    if 'WORD_VOCAB' in globals():
        for norm_key, vocab_info in WORD_VOCAB.items():
            source = vocab_info.get('source', '')
            if 'morpheme_from' in source:
                parent = source.split('morpheme_from:')[-1]
                pairs.append({
                    'input': f'egyptian: {norm_key}', 
                    'target': f'morpheme of: {parent}'
                })
    
    print(f'✅ Dict-LoRA training pairs built: {len(pairs)}')
    return pairs

# ── Global Config ────────────────────────────────────────────────────────────
LORA_BASE_MODEL_ID = 'Helsinki-NLP/opus-mt-de-en' 
_lora_tokenizer = None
_lora_model     = None
_LORA_READY     = False

def load_or_train_lora_dict():
    global _lora_tokenizer, _lora_model, _LORA_READY

    # 1. تحميل الـ Tokenizer
    _lora_tokenizer = AutoTokenizer.from_pretrained(LORA_BASE_MODEL_ID)

    # 2. التحقق من وجود نسخة مدربة مسبقاً
    if os.path.exists(LORA_ADAPTER_PATH):
        print('🚀 Loading cached Dict-LoRA adapter...')
        try:
            base_model = AutoModelForSeq2SeqLM.from_pretrained(
                LORA_BASE_MODEL_ID,
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
            )
            _lora_model = PeftModel.from_pretrained(base_model, LORA_ADAPTER_PATH)
            _lora_model.eval()
            _LORA_READY = True
            print(f'✅ Dict-LoRA adapter loaded from {LORA_ADAPTER_PATH}')
            return
        except Exception as e:
            print(f'⚠️ Failed to load cached adapter: {e}. Retraining...')

    # 3. بناء الموديل للتدريب (عند أول مرة تشغيل)
    print('🏗️ Starting Dict-LoRA training (First time setup)...')
    base = AutoModelForSeq2SeqLM.from_pretrained(
        LORA_BASE_MODEL_ID,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
    )
    
    # التعديل السحري: target_modules="all-linear" يحل مشكلة الـ ValueError
    lora_cfg = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules="all-linear", 
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.SEQ_2_SEQ_LM
    )
    
    _lora_model = get_peft_model(base, lora_cfg)
    _lora_model.print_trainable_parameters()

    # 4. تحضير البيانات
    train_pairs = build_lora_training_data()
    if not train_pairs:
        print("❌ No training data found. Make sure WORD_DICT is loaded.")
        return

    dataset = HFDataset.from_list(train_pairs)

    # التعديل السحري 2: استخدام text_target بدلاً من as_target_tokenizer لحل الـ AttributeError
    def tokenize_fn(ex):
        model_inputs = _lora_tokenizer(ex['input'], max_length=128, truncation=True)
        labels = _lora_tokenizer(text_target=ex['target'], max_length=128, truncation=True)
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    print("📊 Tokenizing dataset...")
    tokenized_ds = dataset.map(tokenize_fn, batched=True, remove_columns=dataset.column_names)

    # 5. إعدادات التدريب (مجهزة للـ Kaggle GPU)
    args = Seq2SeqTrainingArguments(
        output_dir="./lora_results",
        per_device_train_batch_size=16,
        num_train_epochs=1,
        save_strategy="no",
        logging_steps=50,
        learning_rate=3e-4,
        weight_decay=0.01,
        fp16=torch.cuda.is_available(),
        report_to="none"
    )

    trainer = Seq2SeqTrainer(
        model=_lora_model,
        args=args,
        train_dataset=tokenized_ds,
        data_collator=DataCollatorForSeq2Seq(_lora_tokenizer, model=_lora_model),
    )

    # 6. بدء التدريب
    print("🔥 Training in progress...")
    trainer.train()
    
    # 7. الحفظ النهائي
    _lora_model.save_pretrained(LORA_ADAPTER_PATH)
    _lora_model.eval()
    _LORA_READY = True
    print(f'✅ Dict-LoRA trained and saved to {LORA_ADAPTER_PATH}')
    
    # تنظيف الـ Memory
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

# تشغيل الخلية
load_or_train_lora_dict()

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


🏗️ Starting Dict-LoRA training (First time setup)...


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 2,162,688 || all params: 136,068,608 || trainable%: 1.5894
✅ Dict-LoRA training pairs built: 7171
📊 Tokenizing dataset...


Map:   0%|          | 0/7171 [00:00<?, ? examples/s]

🔥 Training in progress...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
50,3.827459
100,3.023904
150,2.952790
200,2.845343
250,2.774207
300,2.691756
350,2.795951
400,2.787192


✅ Dict-LoRA trained and saved to /kaggle/working/dict_lora_adapter_v17


In [28]:
# بدلاً من faiss-gpu، هنستخدم faiss-cpu لأنها الأسهل في التثبيت وموجودة لكل الإصدارات
# وهي سريعة جداً برضه في الـ Indexing للـ Pipeline بتاعنا
!pip install faiss-cpu
!pip install rank-bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 46.0 MB/s eta 0:00:00:00:0100:01


## CELL 6 — Build FAISS + BM25 Indexes (German TLA corpus)

In [29]:
import pickle, faiss, gc
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from rank_bm25 import BM25Okapi

EMBED_MODEL    = None
faiss_index    = None
bbaw_meta      = None
bm25_index     = None
df_test_global = None

if (os.path.exists(FAISS_INDEX_PATH) and
        os.path.exists(FAISS_META_PATH) and
        os.path.exists(BM25_CACHE_PATH)):
    print('Loading cached FAISS + BM25...')
    faiss_index = faiss.read_index(FAISS_INDEX_PATH)
    with open(FAISS_META_PATH, 'rb') as f: bbaw_meta = pickle.load(f)
    with open(BM25_CACHE_PATH, 'rb') as f: bm25_index = pickle.load(f)
    print(f'  FAISS vectors : {faiss_index.ntotal}')
    if os.path.exists(TEST_SET_PATH):
        df_test_global = pd.read_csv(TEST_SET_PATH)
        print(f'  Test rows     : {len(df_test_global)}')
    if bbaw_meta and bbaw_meta.get('translations'):
        print(f'  Sample: {bbaw_meta["translations"][:2]}')
    print('✅ Cached indexes loaded')
else:
    print('Building indexes from scratch...')
    dataset = load_dataset(
        'thesaurus-linguae-aegyptiae/tla-Earlier_Egyptian_original-v18-premium', split='train')
    df_raw  = pd.DataFrame(dataset)
    cols_drop = [c for c in ['hieroglyphs','dateNotBefore','dateNotAfter'] if c in df_raw.columns]
    df_clean  = df_raw.drop(columns=cols_drop)
    df_clean  = df_clean.dropna(subset=['transliteration','translation'])
    df_clean  = df_clean[df_clean['transliteration'].str.strip() != '']
    df_clean  = df_clean[df_clean['translation'].str.strip() != '']
    df_clean  = df_clean.drop_duplicates(subset=['transliteration','translation'])
    df_clean  = df_clean.sample(frac=1, random_state=42).reset_index(drop=True)
    print(f'  Clean records: {len(df_clean)}')
    print(f'  Sample German: {df_clean["translation"].iloc[0]}')
    split_idx = int(len(df_clean) * TRAIN_SPLIT)
    df_train  = df_clean.iloc[:split_idx].copy()
    df_test_global = df_clean.iloc[split_idx:].copy()
    df_test_global.to_csv(TEST_SET_PATH, index=False)
    EMBED_MODEL = SentenceTransformer('BAAI/bge-m3', device='cpu')
    norms   = df_train['transliteration'].apply(normalize_tla).tolist()
    vectors = EMBED_MODEL.encode(norms, batch_size=128, show_progress_bar=True,
                                  normalize_embeddings=True).astype('float32')
    faiss_index = faiss.IndexFlatIP(vectors.shape[1])
    faiss_index.add(vectors)
    faiss.write_index(faiss_index, FAISS_INDEX_PATH)
    bbaw_meta = {'transcriptions': df_train['transliteration'].tolist(),
                  'translations'  : df_train['translation'].tolist(), 'lang': 'de'}
    with open(FAISS_META_PATH, 'wb') as f: pickle.dump(bbaw_meta, f)
    bm25_tokens = [normalize_tla(t).split() for t in df_train['transliteration']]
    bm25_index  = BM25Okapi(bm25_tokens)
    with open(BM25_CACHE_PATH, 'wb') as f: pickle.dump(bm25_index, f)
    print(f'  FAISS vectors : {faiss_index.ntotal}')
    print('✅ Indexes built and cached')

# Re-train LoRA now that corpus is loaded (corpus pairs need FAISS meta)
if not _LORA_READY:
    print('\nRetrying LoRA training with corpus data...')
    load_or_train_lora_dict()

print(f'\n🌍 Corpus language: {bbaw_meta.get("lang", "de")}')

Building indexes from scratch...


README.md: 0.00B [00:00, ?B/s]

train.jsonl:   0%|          | 0.00/5.92M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

  Clean records: 9504
  Sample German: Das ist das Beseitigen der $tḫn$-Verletzung.


...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/74 [00:00<?, ?it/s]

  FAISS vectors : 9408
✅ Indexes built and cached

🌍 Corpus language: de


In [33]:
import safetensors._safetensors_rust as _sr
import transformers.modeling_utils as _mu
from safetensors import safe_open as _orig_safe_open

# ── ركزي هنا: بنرجع الدوال الأصلية لمكانها عشان نمسح أثر الـ Patch ──
_sr.safe_open = _orig_safe_open
_mu.safe_open = _orig_safe_open

print("✅ Patch removed! Original safe_open restored.")

✅ Patch removed! Original safe_open restored.


## CELL 7 — Load Seed-X + Sentiment + spaCy

In [34]:
from transformers import AutoModelForCausalLM, PreTrainedTokenizerFast, pipeline
import torch, spacy, gc, os

# 1. الإعدادات الأساسية
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_PATH = 'ByteDance-Seed/Seed-X-PPO-7B' if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ else 'seed-x-ppo-7b'

# 2. تنظيف الذاكرة بعمق
if 'seed_x_model' in globals(): del seed_x_model
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f'📥 Loading Seed-X from: {MODEL_PATH} on {DEVICE}...')

# 3. التحميل (النسخة النضيفة)
seed_x_tokenizer = PreTrainedTokenizerFast.from_pretrained(MODEL_PATH, trust_remote_code=True)
if seed_x_tokenizer.pad_token is None: 
    seed_x_tokenizer.pad_token = seed_x_tokenizer.eos_token

seed_x_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, 
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
    device_map='auto', 
    trust_remote_code=True, 
    low_cpu_mem_usage=True
)
seed_x_model.eval()

print('✅ Seed-X Loaded Successfully!')

# 4. الأدوات المساعدة
if 'sentiment_pipe' not in globals():
    sentiment_pipe = pipeline('sentiment-analysis', 
                              model='cardiffnlp/twitter-roberta-base-sentiment-latest',
                              device=0 if DEVICE=='cuda' else -1)

if 'nlp_spacy' not in globals():
    try: nlp_spacy = spacy.load('en_core_web_sm')
    except: 
        os.system('python -m spacy download en_core_web_sm')
        nlp_spacy = spacy.load('en_core_web_sm')

print('✅ Pipeline v17.2 Ready!')

📥 Loading Seed-X from: ByteDance-Seed/Seed-X-PPO-7B on cpu...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

✅ Seed-X Loaded Successfully!


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

✅ Pipeline v17.2 Ready!


## CELL 8 — Core Helpers v17
### FIX 1 (critical): assemble_words now uses determinatives as HARD WORD BOUNDARIES
### Each phoneme-group between determinatives is a separate token fed to the DP assembler

In [36]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 29.1 MB/s eta 0:00:0000:0100:01


In [37]:
from rapidfuzz import fuzz
import time

STANDALONE_PARTICLES = {'n','m','r','in','iw','nn','hr','\u1e25r'}

def _get_phonetic(code: str, use_tla: bool = True) -> tuple:
    key  = code.strip().lower()
    info = GARDINER_MAP.get(key, {})
    if info.get('is_determinative', False): return '', True
    ph = (info.get('phonetic_tla','') or info.get('phonetic','')) if use_tla \
         else (info.get('phonetic','') or info.get('phonetic_tla',''))
    return (ph if ph else key), False

# ── FIX 1: Correct word assembly ──────────────────────────────────────────
# Root cause of ḫrpḥrib bug:
# Old code: assemble_words() returned ['ḫrpḥrib'] because it was joining ALL
# phonemes into one word unless a determinative appeared between them.
# For S42 D2 F34 — there is NO determinative, so it fused all three.
#
# Fix: Use the Gardiner sign categories properly.
# S42 = sceptre (phonetic: ḫrp), D2 = face (phonetic: ḥr), F34 = heart (phonetic: ib)
# These are THREE separate phonetic signs → three separate morpheme tokens.
# We must NOT join consecutive phonetic signs into one word automatically.
# The word_assembly_result() DP handles merging decisions.

def assemble_words(codes: list, use_tla: bool = True) -> list:
    """
    v17 FIX: Returns one token per non-determinative sign (or particle).
    Determinatives reset the word boundary.
    The word assembly DP (assemble_words_v11) then decides how to MERGE tokens.
    """
    words        = []
    current_word = []
    prev_was_det = True

    for c in codes:
        phonetic, is_det = _get_phonetic(c, use_tla=use_tla)
        if is_det:
            # Flush current word segment
            if current_word:
                words.append(''.join(current_word))
                current_word = []
            prev_was_det = True
        else:
            if phonetic:
                if prev_was_det and phonetic.lower() in STANDALONE_PARTICLES and not current_word:
                    # Standalone particle (n, m, r...) — emit as its own word
                    words.append(phonetic)
                    prev_was_det = False
                else:
                    current_word.append(phonetic)
                    prev_was_det = False

    if current_word: words.append(''.join(current_word))
    return [w for w in words if w]

def assemble_phoneme_tokens(codes: list, use_tla: bool = True) -> list:
    """
    FIX 1 (new): Returns ONE TOKEN PER SIGN (not fused).
    This is what gets fed to word_assembly_result() so the DP can decide
    how to split/merge. Determinatives still act as hard boundaries.

    Example: S42 D2 F34 → ['ḫrp', 'ḥr', 'ib']  (3 separate tokens)
    Previously: → ['ḫrpḥrib']  (wrong — fused before DP sees it)
    """
    tokens       = []
    current_word = []
    prev_was_det = True

    for c in codes:
        phonetic, is_det = _get_phonetic(c, use_tla=use_tla)
        if is_det:
            # Hard boundary: flush accumulated phonemes as separate tokens
            tokens.extend(current_word)
            current_word = []
            prev_was_det = True
        else:
            if phonetic:
                if prev_was_det and phonetic.lower() in STANDALONE_PARTICLES:
                    # Flush then emit particle as own token
                    tokens.extend(current_word)
                    current_word = []
                    tokens.append(phonetic)
                    prev_was_det = False
                else:
                    # Add to current word group but keep as SEPARATE token
                    current_word.append(phonetic)
                    prev_was_det = False

    tokens.extend(current_word)
    return [t for t in tokens if t]

def assemble_tla(codes):      return ' '.join(assemble_words(codes, use_tla=True))
def assemble_standard(codes): return ' '.join(assemble_words(codes, use_tla=False))

def assemble_raw_with_det(codes):
    parts, current = [], []
    for c in codes:
        ph, is_det = _get_phonetic(c, use_tla=True)
        if is_det:
            if current: parts.append(''.join(current)); current = []
            parts.append(f'[{c.upper()}]')
        elif ph: current.append(ph)
    if current: parts.append(''.join(current))
    return ' '.join(parts)

def get_meanings(codes):
    return [GARDINER_MAP.get(c.strip().lower(),{}).get('meaning','')
            for c in codes
            if not GARDINER_MAP.get(c.strip().lower(),{}).get('is_determinative',False)
            and GARDINER_MAP.get(c.strip().lower(),{}).get('meaning','')]

def get_embed_model():
    global EMBED_MODEL
    if EMBED_MODEL is None: EMBED_MODEL = SentenceTransformer('BAAI/bge-m3', device='cpu')
    return EMBED_MODEL

def fix_grammar(text):
    doc = nlp_spacy(text)
    sents = [s.text.strip() for s in doc.sents]
    return ' '.join(s[0].upper() + s[1:] for s in sents if s)

def get_sentiment(text):
    raw = sentiment_pipe(text[:512])[0][0]
    label = raw['label'].lower()
    return {'label': {'positive':'Positive','neutral':'Neutral','negative':'Negative'}.get(label,label),
            'confidence': raw['score']}

def translate_seedx(text, src_lang, tgt_lang, max_new_tokens=150):
    if not text.strip(): return ''
    tgt_tag = LANG_TAGS.get(tgt_lang, 'en')
    prompt  = f'Translate the following text into {tgt_lang}:\n{text}\n<{tgt_tag}>'
    inputs  = seed_x_tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)
    inputs.pop('token_type_ids', None)
    dev = next(seed_x_model.parameters()).device
    inputs = {k: v.to(dev) for k, v in inputs.items()}
    with torch.no_grad():
        out = seed_x_model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            num_beams=2, do_sample=False, no_repeat_ngram_size=3, repetition_penalty=1.4,
            pad_token_id=seed_x_tokenizer.pad_token_id,
            eos_token_id=seed_x_tokenizer.eos_token_id,
        )
    new_tok = out[0][inputs['input_ids'].shape[1]:]
    decoded = seed_x_tokenizer.decode(new_tok, skip_special_tokens=True).strip()
    parts   = re.split(r'[\n]|(?<=[.!?])\s', decoded)
    clean   = parts[0].strip() if parts else decoded
    return re.sub(r'[^\w\s\u0600-\u06FF.,!?\'"-]','',clean).strip()

def translate_arabic_verified(text_de, max_retries=2):
    arabic  = translate_seedx(text_de, 'German', 'Arabic', max_new_tokens=200)
    back_de = translate_seedx(arabic, 'Arabic', 'German',  max_new_tokens=150)
    orig_w  = set(text_de.lower().split())
    back_w  = set(back_de.lower().split())
    overlap = len(orig_w & back_w) / max(len(orig_w), 1)
    return {'arabic': arabic, 'verified': overlap >= 0.25,
            'back_translation': back_de, 'confidence': round(overlap, 3)}

def interpret_rag_score(score, k=RRF_K, alpha=HYBRID_ALPHA):
    ms = 1.0 / (k + 1); r = score / ms if ms > 0 else 0
    if r >= 0.90: return f'✅ Excellent  ({score:.4f} = {r:.0%} of max {ms:.4f})'
    if r >= 0.60: return f'👍 Good       ({score:.4f} = {r:.0%} of max {ms:.4f})'
    if r >= 0.30: return f'⚠️  Weak       ({score:.4f} = {r:.0%} of max {ms:.4f})'
    return f'❌ No match   ({score:.4f} = {r:.0%} of max {ms:.4f})'

# ── FIX 1 smoke test ───────────────────────────────────────────────────────
_test_codes = ['S42','D2','F34']
_tokens     = assemble_phoneme_tokens(_test_codes)
print('\nFIX 1 test: S42 D2 F34')
print(f'  assemble_phoneme_tokens → {_tokens}')
print(f'  Expected               : ["ḫrp", "ḥr", "ib"] or equivalent phonetics')
print('\n✅ Core helpers ready (v17)')


FIX 1 test: S42 D2 F34
  assemble_phoneme_tokens → ['ḫrp', 'ḥr', 'ib']
  Expected               : ["ḫrp", "ḥr", "ib"] or equivalent phonetics

✅ Core helpers ready (v17)


## CELL 9 — Hybrid Search v17 + FIX 3: Royal Name Penalty
### FIX 3: After RRF, detect if top hit is a proper noun (royal name) while query is a verb phrase
### If so, penalise that hit and surface the next best semantic match

In [39]:
import re as _re_rag

# Heuristics for detecting proper-noun / royal-name translations
# German royal names are typically capitalized single words with no verb
_ROYAL_NAME_RE = _re_rag.compile(
    r'^[A-ZÄÖÜ][a-zäöüß]+\.?$|'
    r'^(Pharao|König|Königin|Prinz|Prinzessin|Ramses|Thutmose|Amenhotep|Senusret|Sesostris|'
    r'Nefertiti|Kleopatra|Hatschepsut|Tutanchamun|Echnaton|Sethos|Merenptah|Seti|Senweseret)',
    _re_rag.IGNORECASE
)

def _is_proper_noun_translation(translation: str) -> bool:
    """FIX 3: Returns True if the German translation looks like a proper noun/royal name."""
    t = translation.strip()
    # Single capitalized word (name)
    if _re_rag.match(r'^[A-ZÄÖÜ][a-zäöüßA-ZÄÖÜ-]+\.?$', t): return True
    # Known royal name patterns
    if _ROYAL_NAME_RE.search(t): return True
    return False

def _query_looks_like_verb_phrase(assembled_words: list) -> bool:
    """FIX 3: Returns True if the assembled tokens suggest a verb phrase not a name."""
    if not assembled_words: return False
    # Verb phrases tend to have 3+ tokens, or contain particles like n/m/r
    if len(assembled_words) >= 3: return True
    for w in assembled_words:
        if w.lower() in STANDALONE_PARTICLES: return True
    return False

def _rerank_results(results: list, assembled_words: list) -> list:
    """
    FIX 3: If query looks like a verb phrase and top hits are proper nouns,
    penalize those hits so the correct semantic match rises to the top.
    """
    if not _query_looks_like_verb_phrase(assembled_words): return results
    reranked = []
    for r in results:
        score = r['score']
        if _is_proper_noun_translation(r.get('translation', '')):
            score = score * ROYAL_NAME_PENALTY
        reranked.append({**r, 'score': score, 'rrf_score': score})
    reranked.sort(key=lambda x: x['score'], reverse=True)
    return reranked

def _dense_search(query_norm, top_k):
    model = get_embed_model()
    q_vec = model.encode([query_norm], normalize_embeddings=True).astype('float32')
    if hasattr(faiss_index, 'nprobe'): faiss_index.nprobe = 10
    D, I = faiss_index.search(q_vec, top_k * 2)
    return [{'idx':int(idx),'score':float(score),
              'transcription':bbaw_meta['transcriptions'][idx],
              'translation':bbaw_meta['translations'][idx]}
            for score, idx in zip(D[0], I[0]) if idx >= 0]

def _bm25_search(query_norm, top_k):
    tokens      = query_norm.split()
    bm25_scores = bm25_index.get_scores(tokens)
    top_indices = np.argsort(bm25_scores)[::-1][:top_k * 2]
    return [{'idx':int(idx),'score':float(bm25_scores[idx]),
              'transcription':bbaw_meta['transcriptions'][idx],
              'translation':bbaw_meta['translations'][idx]}
            for idx in top_indices if bm25_scores[idx] > 0]

def _rrf(dense, sparse, k=RRF_K, alpha=HYBRID_ALPHA):
    scores = {}
    for rank, r in enumerate(dense): scores[r['idx']] = scores.get(r['idx'],0) + alpha*(1/(k+rank+1))
    for rank, r in enumerate(sparse): scores[r['idx']] = scores.get(r['idx'],0) + (1-alpha)*(1/(k+rank+1))
    all_hits = {r['idx']: r for r in dense + sparse}
    return [{'rrf_score':sc,'score':sc,**all_hits[idx]}
            for idx, sc in sorted(scores.items(), key=lambda x:-x[1])]

def _build_search_queries(tla_tokens, codes):
    queries = set()
    q1 = normalize_tla(''.join(tla_tokens));  queries.add(q1) if q1 else None
    q2 = normalize_tla(' '.join(tla_tokens)); queries.add(q2) if q2 else None
    if codes:
        phonetics = []
        for c in codes:
            info = GARDINER_MAP.get(c.strip().lower(), {})
            ph = info.get('phonetic_tla','') or info.get('phonetic','')
            if ph and not info.get('is_determinative',False): phonetics.append(ph)
        if phonetics:
            q3 = normalize_tla(' '.join(phonetics));  queries.add(q3) if q3 else None
            q3h= normalize_tla('-'.join(phonetics)); queries.add(q3h) if q3h else None
    return [q for q in queries if q and q.strip()]

def hybrid_search(query_raw, top_k=TOP_K_RAG, tla_tokens=None, codes=None, assembled_words=None):
    queries = _build_search_queries(tla_tokens, codes or []) if tla_tokens else [normalize_tla(query_raw)]
    if not queries: queries = [normalize_tla(query_raw)]
    dense_all, sparse_all = [], []
    for q in queries:
        dense_all.extend(_dense_search(q, top_k))
        sparse_all.extend(_bm25_search(q, top_k))
    dense_best  = {}; sparse_best = {}
    for r in dense_all:
        if r['idx'] not in dense_best or r['score'] > dense_best[r['idx']]['score']: dense_best[r['idx']] = r
    for r in sparse_all:
        if r['idx'] not in sparse_best or r['score'] > sparse_best[r['idx']]['score']: sparse_best[r['idx']] = r
    merged = _rrf(list(dense_best.values()), list(sparse_best.values()))[:top_k]
    # FIX 3: Apply royal name penalty
    if assembled_words:
        merged = _rerank_results(merged, assembled_words)
    return merged

print('✅ Hybrid Search v17 ready (+ royal-name reranking)')

✅ Hybrid Search v17 ready (+ royal-name reranking)


## CELL 10 — RAG + German Output v17
### FIX 4: Semantic guard — skip top RAG hit if it's a proper noun AND confidence gap is large
### FIX 6: Improved fallback chain with better German phrase construction

In [40]:
import re as _re

_LLM_GARBAGE_PATTERNS = [
    r'^Output\b',r'^These are',r'^Translation[:\s]',r'^German\s*[Tt]ranslation[:\s]',
    r'^Translate\b',r'^you are (a|an|the)\b',r'^i am (a|an|the)\b',r'\<[a-z]{2}\>$',
    r'^OPINION\b',r'^COT\b',r'^THOUGHTS\b',r'^This is a translation',r'hieroglyphic',r'^An in-depth analysis',
]
_GARBAGE_RE = _re.compile('|'.join(_LLM_GARBAGE_PATTERNS), _re.IGNORECASE)

def _clean_llm_output(raw):
    if not raw: return ''
    lines = [ln.strip() for ln in raw.split('\n')]
    clean = []
    for ln in lines:
        if not ln: continue
        ln = _re.sub(r'\s*<[a-z]{2}>\s*$', '', ln).strip()
        if not ln: continue
        if _GARBAGE_RE.search(ln): continue
        clean.append(ln)
    return clean[0] if clean else raw.strip()

def _count_tokens(text): return len(text) // 4

def _get_best_german_from_rag(rag_results, assembled_words=None):
    """
    FIX 4: If query is a verb phrase, skip proper-noun-only hits.
    Try the first hit that is NOT a proper noun.
    Fall back to the top hit if none found.
    """
    is_verb_phrase = _query_looks_like_verb_phrase(assembled_words or [])
    for r in rag_results:
        t = r.get('translation', '').strip()
        if len(t.replace('.','').replace(' ','')) < 10: continue
        if is_verb_phrase and _is_proper_noun_translation(t):
            continue   # Skip proper nouns when query is a verb phrase
        return t
    # Fallback: return top hit regardless
    for r in rag_results:
        t = r.get('translation', '').strip()
        if len(t.replace('.','').replace(' ','')) > 10: return t
    return ''

def enrich_meanings_with_lora(assembled_words):
    enriched = {}
    for word in assembled_words:
        cands = find_candidates(word.lower())
        trie_meaning = cands[0]['meaning'] if cands else ''
        if not trie_meaning and _LORA_READY:
            enriched[word] = dict_lora_lookup(word)
        else:
            enriched[word] = trie_meaning
    return enriched

def german_to_english_strict(german_text, max_new_tokens=150):
    if not german_text or not german_text.strip(): return ''
    prompt = (f'Translate from German to English. Output only the translation.\n'
              f'German: {german_text}\nEnglish:')
    inputs = seed_x_tokenizer(prompt, return_tensors='pt', truncation=True, max_length=256)
    inputs.pop('token_type_ids', None)
    dev = next(seed_x_model.parameters()).device
    inputs = {k: v.to(dev) for k, v in inputs.items()}
    with torch.no_grad():
        out = seed_x_model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            num_beams=2, do_sample=False, no_repeat_ngram_size=3, repetition_penalty=1.4,
            pad_token_id=seed_x_tokenizer.pad_token_id, eos_token_id=seed_x_tokenizer.eos_token_id,
        )
    raw = seed_x_tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    cleaned = _clean_llm_output(raw)
    return cleaned if cleaned and set(cleaned) > set('. -') else german_text

def translate_with_rag_prompt(query_original, query_norm, retrieved, meanings, assembled_words=None):
    """
    v17 FIX 4 + 6: Proper-noun-aware RAG retrieval + improved fallback chain.
    """
    if assembled_words is None: assembled_words = query_norm.split()

    # Step 1: German from RAG (with proper noun guard)
    best_german = _get_best_german_from_rag(retrieved, assembled_words)
    if best_german: return best_german

    # Step 2: Dict-LoRA enrichment
    lora_meanings = enrich_meanings_with_lora(assembled_words)
    non_empty = [v for v in lora_meanings.values() if v]
    if non_empty: return ', '.join(non_empty)

    # Step 3: Gardiner sign meanings as German phrase
    if meanings:
        return ', '.join(m for m in meanings if m)

    # Step 4: LLM fallback with German prompt
    meaning_hint = ', '.join(m for m in meanings if m)
    header = (
        'Du bist ein Experte für Altägyptisch. '
        'Übersetze die Transliteration ins Deutsche. '
        'Gib NUR die deutsche Übersetzung aus. Keine Erklärungen.\n\n'
        f'Transliteration: {query_original}\n'
        f'Wörter: {" ".join(assembled_words)}\n'
        f'Zeichenbedeutungen: {meaning_hint or "N/A"}\n\n'
        'Ähnlichste Corpus-Beispiele:\n'
    )
    footer = '\nDeutsche Übersetzung:'
    budget = _count_tokens(header) + _count_tokens(footer)
    examples_text = ''
    for i, ex in enumerate(retrieved, 1):
        block = f'\n{i}. [{ex["transcription"]}] -> {ex["translation"]}'
        if budget + _count_tokens(block) > MAX_PROMPT_TOKENS: break
        examples_text += block; budget += _count_tokens(block)
    prompt = header + examples_text + footer
    inputs = seed_x_tokenizer(prompt, return_tensors='pt', truncation=True, max_length=4096)
    inputs.pop('token_type_ids', None)
    dev = next(seed_x_model.parameters()).device
    inputs = {k: v.to(dev) for k, v in inputs.items()}
    with torch.no_grad():
        out = seed_x_model.generate(
            **inputs, max_new_tokens=200, num_beams=2, do_sample=False,
            no_repeat_ngram_size=3, repetition_penalty=1.4,
            pad_token_id=seed_x_tokenizer.pad_token_id, eos_token_id=seed_x_tokenizer.eos_token_id,
        )
    new_tok = out[0][inputs['input_ids'].shape[1]:]
    cleaned = _clean_llm_output(seed_x_tokenizer.decode(new_tok, skip_special_tokens=True).strip())
    return cleaned if cleaned else (meaning_hint or query_original)

print('✅ RAG prompt builder ready (v17 — proper-noun guard)')

✅ RAG prompt builder ready (v17 — proper-noun guard)


## CELL 10.5 — Translation Cache

In [41]:
import hashlib
import pickle as _pickle

_TRANSLATION_CACHE      : dict = {}
_TRANSLATION_CACHE_PATH = os.path.join(_WORK_ROOT, 'translation_cache_v17.pkl')

def _load_translation_cache():
    global _TRANSLATION_CACHE
    if os.path.exists(_TRANSLATION_CACHE_PATH):
        try:
            with open(_TRANSLATION_CACHE_PATH, 'rb') as f: _TRANSLATION_CACHE = _pickle.load(f)
            print(f'✅ Translation cache: {len(_TRANSLATION_CACHE)} entries')
        except: _TRANSLATION_CACHE = {}; print('⚠️  Cache corrupted — starting fresh')
    else: print('   Translation cache: empty')

def _save_translation_cache():
    try:
        with open(_TRANSLATION_CACHE_PATH, 'wb') as f: _pickle.dump(_TRANSLATION_CACHE, f)
    except Exception as e: print(f'⚠️  Cache save failed: {e}')

_load_translation_cache()

_german_to_english_strict_original = german_to_english_strict
def german_to_english_strict(german_text, max_new_tokens=150):
    if not german_text or not german_text.strip(): return ''
    ck = hashlib.md5(german_text.strip().encode('utf-8')).hexdigest()
    if ck in _TRANSLATION_CACHE: return _TRANSLATION_CACHE[ck]
    result = _german_to_english_strict_original(german_text, max_new_tokens)
    _TRANSLATION_CACHE[ck] = result; _save_translation_cache()
    return result

print('✅ Translation cache active')

   Translation cache: empty
✅ Translation cache active


## CELL 11 — Path Handlers v17

In [42]:
def sentence_path(codes, rag_results, meanings, assembled_words, include_english=False):
    query_orig = ' '.join(assembled_words)
    query_norm = normalize_tla(query_orig)

    german = translate_with_rag_prompt(
        query_orig, query_norm, rag_results, meanings,
        assembled_words=assembled_words
    )

    if not german or set(german.strip()) <= set('. -') or len(german.strip()) < 3:
        lm = enrich_meanings_with_lora(assembled_words)
        german = ', '.join(v for v in lm.values() if v) or \
                 ', '.join(m for m in meanings if m) or query_orig

    english = ''
    if include_english:
        english = fix_grammar(german_to_english_strict(german))
        if not english or set(english.strip()) <= set('. -'): english = german

    ar_result = translate_arabic_verified(german)
    best      = rag_results[0]
    return {
        'path': 'sentence', 'german': german, 'english': english,
        'arabic': ar_result['arabic'], 'arabic_verified': ar_result['verified'],
        'arabic_confidence': ar_result['confidence'],
        'sentiment': get_sentiment(german), 'intention': get_intention(german),
        'rag_score': best['score'], 'rag_source': best['transcription'],
        'rag_results': rag_results, 'assembled_words': assembled_words,
        'german_source': german,
    }

print('✅ Path handlers ready (v17)')

✅ Path handlers ready (v17)


## CELL 12 — Full Pipeline v17
### FIX 1 applied here: use assemble_phoneme_tokens() for word assembly DP input
### so S42+D2+F34 → ['ḫrp','ḥr','ib'] → DP merges into 'ḫrp-ḥr-ib' → RAG finds 'Leiter der Mitte'

In [44]:
# ── FIX: Connecting Word Assembly to LoRA ──────────────────────────────────

def dict_lora_lookup(egyptian_word: str) -> str:
    """Uses the fine-tuned LoRA model to predict meanings for unknown words."""
    global _lora_model, _lora_tokenizer, _LORA_READY
    
    if not _LORA_READY or _lora_model is None:
        return ""
    
    # تحضير النص بنفس التنسيق اللي الموديل اتدرب عليه في Cell 3.6
    prompt = f"egyptian: {egyptian_word}"
    inputs = _lora_tokenizer(prompt, return_tensors="pt", truncation=True).to(_lora_model.device)
    
    with torch.no_grad():
        outputs = _lora_model.generate(
            **inputs, 
            max_new_tokens=50,
            num_beams=2,
            do_sample=False,
            pad_token_id=_lora_tokenizer.pad_token_id,
            eos_token_id=_lora_tokenizer.eos_token_id
        )
    
    return _lora_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

def enrich_meanings_with_lora(assembled_words: list) -> dict:
    """Iterates through words and asks LoRA for help if the dictionary is silent."""
    enriched = {}
    for word in assembled_words:
        # بنحاول ندور في الـ Trie اللي عندك الأول (لو متعرف)
        trie_meaning = ""
        if 'get_trie_candidates' in globals():
            cands = get_trie_candidates(word)
            trie_meaning = cands[0]['meaning'] if cands else ""
        
        # لو مفيش معنى في القاموس، بنروح للـ LoRA "المنقذ"
        if not trie_meaning and _LORA_READY:
            enriched[word] = dict_lora_lookup(word)
        else:
            enriched[word] = trie_meaning
            
    return enriched

# دالة مساعدة عشان الـ Pipeline يكمل لو الـ RAG فشل
def _get_best_german_from_rag(rag_results, assembled_words=None):
    if not rag_results: return ""
    # بناخد أول نتيجة (الأعلى تشابهاً)
    return rag_results[0].get('translation', '')

print("✅ Dict-LoRA lookup tools defined. Ready for Smoke Tests!")

✅ Dict-LoRA lookup tools defined. Ready for Smoke Tests!


In [47]:
import re
import torch

# ── FIX: Define Language Tags & Translation Helpers ───────────────────────

# القاموس اللي كان ناقص في الـ Traceback
LANG_TAGS = {
    'English': 'en',
    'German': 'de',
    'Arabic': 'ar',
    'French': 'fr',
    'Italian': 'it',
    'Spanish': 'es'
}

def translate_seedx(text, src_lang, tgt_lang, max_new_tokens=150):
    """Core translation function using Seed-X model."""
    if not text.strip(): return ''
    
    # التأكد من وجود التاغ، الافتراضي إنجليزي لو مش موجود
    tgt_tag = LANG_TAGS.get(tgt_lang, 'en')
    prompt  = f'Translate the following text into {tgt_lang}:\n{text}\n<{tgt_tag}>'
    
    inputs  = seed_x_tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)
    inputs.pop('token_type_ids', None)
    
    # نقل البيانات للـ Device (GPU/CPU)
    dev = next(seed_x_model.parameters()).device
    inputs = {k: v.to(dev) for k, v in inputs.items()}
    
    with torch.no_grad():
        out = seed_x_model.generate(
            **inputs, 
            max_new_tokens=max_new_tokens,
            num_beams=2, 
            do_sample=False, 
            no_repeat_ngram_size=3, 
            repetition_penalty=1.4,
            pad_token_id=seed_x_tokenizer.pad_token_id,
            eos_token_id=seed_x_tokenizer.eos_token_id,
        )
    
    new_tok = out[0][inputs['input_ids'].shape[1]:]
    decoded = seed_x_tokenizer.decode(new_tok, skip_special_tokens=True).strip()
    
    # تنظيف المخرجات من أي حروف غريبة
    parts = re.split(r'[\n]|(?<=[.!?])\s', decoded)
    clean = parts[0].strip() if parts else decoded
    return re.sub(r'[^\w\s\u0600-\u06FF.,!?\'"-]', '', clean).strip()

def translate_arabic_verified(text_de, max_retries=2):
    """Translates to Arabic and verifies via back-translation."""
    arabic  = translate_seedx(text_de, 'German', 'Arabic', max_new_tokens=200)
    back_de = translate_seedx(arabic, 'Arabic', 'German', max_new_tokens=150)
    
    # حساب نسبة التشابه (Word Overlap) للتحقق من الجودة
    orig_w  = set(text_de.lower().split())
    back_w  = set(back_de.lower().split())
    overlap = len(orig_w & back_w) / max(len(orig_w), 1)
    
    return {
        'arabic': arabic, 
        'verified': overlap >= 0.25,
        'back_translation': back_de, 
        'confidence': round(overlap, 3)
    }

print("✅ LANG_TAGS defined and translation functions updated!")

✅ LANG_TAGS defined and translation functions updated!


In [ ]:
def full_pipeline(gardiner_input: str, verbose: bool = True, include_english: bool = False) -> dict:
    t0 = time.time()

    # ── Step 1: Parse + code snapping ───────────────────────────────────────
    raw_codes = gardiner_input.strip().upper().split()
    codes, snap_log = normalize_codes(raw_codes)
    if verbose and snap_log: print(f'⚠️  Code snapping: {snap_log}')

    # ── Step 2: Assemble words (high-level, for display + RAG query) ────────
    assembled_words = assemble_words(codes, use_tla=True)
    tla_query       = ' '.join(assembled_words)
    tla_norm        = normalize_tla(tla_query)
    meanings        = get_meanings(codes)

    # ── FIX 1: Phoneme tokens for word assembly DP ──────────────────────────
    # Use individual sign phonetics as tokens → DP decides how to merge
    phoneme_tokens = assemble_phoneme_tokens(codes, use_tla=True)

    if verbose:
        print(f'Input codes     : {raw_codes}')
        if snap_log: print(f'After snapping  : {codes}')
        print(f'Raw + det       : {assemble_raw_with_det(codes)}')
        print(f'Phoneme tokens  : {phoneme_tokens}   ← FIX1: per-sign tokens')
        print(f'TLA query       : {tla_query}')
        print(f'TLA norm        : {tla_norm}')

    # ── Step 3: Word Assembly DP on phoneme tokens ──────────────────────────
    wa = word_assembly_result(phoneme_tokens)

    if verbose:
        print(f'Word Assembly   : {wa["best_segmentation"]}')
        print(f'Assembly Score  : {wa["best_score"]}')
        print(f'Assembly types  : {[m["type"] for m in wa["best_metadata"]]}')
        if wa['alternatives']: print(f'Alternatives    : {wa["alternatives"][:2]}')

    # ── Step 4: Dict-LoRA enrichment ─────────────────────────────────────────
    lora_meanings = enrich_meanings_with_lora(wa['best_segmentation'])
    if verbose:
        non_empty_lora = {w: m for w, m in lora_meanings.items() if m}
        if non_empty_lora: print(f'Dict-LoRA       : {non_empty_lora}')

    # ── Step 5: Hybrid RAG (with assembled_words for proper-noun reranking) ──
    rag_results = hybrid_search(
        tla_query, top_k=TOP_K_RAG,
        tla_tokens=assembled_words, codes=codes,
        assembled_words=assembled_words,   # FIX 3: pass for reranking
    )
    best_score = rag_results[0]['score'] if rag_results else 0.0

    if verbose:
        print(f'RAG best score  : {best_score:.4f} → {interpret_rag_score(best_score)}')
        print(f'Threshold       : {SIMILARITY_THRESHOLD:.4f}')
        if rag_results:
            top = rag_results[0]
            is_pn = _is_proper_noun_translation(top.get('translation',''))
            print(f'Top RAG hit     : [{top["transcription"]}] → {top["translation"]}'
                  f'  {"[proper noun — penalised]" if is_pn and _query_looks_like_verb_phrase(assembled_words) else ""}')

    # ── Step 6: Route ────────────────────────────────────────────────────────
    if best_score >= SIMILARITY_THRESHOLD:
        if verbose: print('Path: SENTENCE  ← RAG succeeded')
        result = sentence_path(codes, rag_results, meanings, assembled_words, include_english=include_english)
    else:
        if verbose: print('Path: FALLBACK  ← below threshold')
        lora_ne    = [v for v in lora_meanings.values() if v]
        word_meats = [m for m in meanings if m]
        per_word   = [m['meaning'] for m in wa['best_metadata'] if m.get('meaning')]

        if lora_ne:   german = ', '.join(lora_ne)
        elif per_word: german = ' '.join(per_word)
        else:
            bg = _get_best_german_from_rag(rag_results, assembled_words)
            german = bg if bg else (f'Altägyptische Zeichen: {", ".join(word_meats)}.' if word_meats else tla_query)

        english = ''
        if include_english:
            english = fix_grammar(german_to_english_strict(german))
            if not english or set(english.strip()) <= set('. -'): english = german

        ar_result = translate_arabic_verified(german)
        result = {
            'path': 'fallback', 'german': german, 'english': english,
            'arabic': ar_result['arabic'], 'arabic_verified': ar_result['verified'],
            'arabic_confidence': ar_result['confidence'],
            'assembled_words': assembled_words,
            'sentiment': get_sentiment(german), 'intention': get_intention(german),
            'rag_score': best_score,
            'rag_source': rag_results[0]['transcription'] if rag_results else '',
            'rag_results': rag_results,
        }

    result.update({
        'input': gardiner_input, 'codes_raw': raw_codes, 'codes_snapped': codes,
        'snap_log': snap_log, 'tla_phonemes': tla_query, 'tla_normalised': tla_norm,
        'phoneme_tokens': phoneme_tokens,
        'meanings': meanings, 'assembly_metadata': wa['best_metadata'],
        'assembled_words': result.get('assembled_words', assembled_words),
        'assembly_alts': wa['alternatives'], 'assembly_score': wa['best_score'],
        'lora_meanings': lora_meanings, 'processing_time': round(time.time()-t0, 2),
    })

    if verbose:
        print('=' * 60)
        print(f'PATH            : {result["path"].upper()}')
        print(f'Phoneme tokens  : {phoneme_tokens}')
        print(f'Word Assembly   : {wa["best_segmentation"]}')
        print(f'TLA norm        : {tla_norm}')
        print(f'German          : {result["german"]}')
        if result.get('english'): print(f'English         : {result["english"]}')
        print(f'Arabic          : {result["arabic"]}')
        if result.get('arabic_verified') is not None:
            print(f'Arabic check    : {"verified" if result["arabic_verified"] else "low conf"} ({result.get("arabic_confidence",0):.0%})')
        s = result.get('sentiment')
        if s: print(f'Sentiment       : {s["label"]} ({s["confidence"]:.0%})')
        i = result.get('intention')
        if i: print(f'Intention       : {i["intention_en"]} / {i["intention_ar"]}')
        print(f'RAG Score       : {result["rag_score"]:.4f}  →  {interpret_rag_score(result["rag_score"])}')
        print(f'Time            : {result["processing_time"]}s')
        print('=' * 60)
    return result


# ── SMOKE TESTS v17 ──────────────────────────────────────────────────────────
print('\n=== SMOKE TESTS v17 ===')

print('\n[Test 1] S42 D2 F34 — expect: Leiter der Mitte')
print('FIX 1 check: phoneme_tokens should be [ḫrp, ḥr, ib] not [ḫrpḥrib]')
r1 = full_pipeline('S42 D2 F34')

print('\n[Test 2] Injury for Horus — expect: Verletzung für den der Horus verletzt')
print('FIX 3 check: should NOT output Senweseret')
r2 = full_pipeline('S29 M17 G1 V13 D57 N35 S29 M17 D36 V13 D57 G5')

print('\n[Test 3] Code snapping — A01 G05 N35 should snap to A1 G5 N35')
r3 = full_pipeline('A01 G05 N35')
print(f'Snap log: {r3["snap_log"]}')
print(f'FIX 2 check: should be A01→A1 G05→G5, not A01→A51 G05→G15')


=== SMOKE TESTS v17 ===

[Test 1] S42 D2 F34 — expect: Leiter der Mitte
FIX 1 check: phoneme_tokens should be [ḫrp, ḥr, ib] not [ḫrpḥrib]
Input codes     : ['S42', 'D2', 'F34']
Raw + det       : ḫrpḥrib
Phoneme tokens  : ['ḫrp', 'ḥr', 'ib']   ← FIX1: per-sign tokens
TLA query       : ḫrpḥrib
TLA norm        : khrphrib
Word Assembly   : ['ḫrpḥr', 'ib']
Assembly Score  : 4.9
Assembly types  : ['fuzzy', 'exact']
Alternatives    : [['ḫrp', 'ḥr', 'ib'], ['ḫrp', 'ḥrib']]
Dict-LoRA       : {'ḫrpḥr': 'morpheme of: jm(,j)-n-nb-nb-nb-nb-nb-nb-nb-nb-nb-nb-nb-nb', 'ib': 'morpheme of: m-ib'}
RAG best score  : 10.4507 → ✅ Excellent  (10.4507 = 63749% of max 0.0164)
Threshold       : 0.0082
Top RAG hit     : [ḫrp-ḥr(.ꞽ)-ꞽb] → Leiter der Mitte  
Path: SENTENCE  ← RAG succeeded


## CELL 13 — Full Evaluation Suite v17

In [ ]:
import nltk
from nltk.translate.bleu_score   import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score                 import rouge_scorer
from sacrebleu.metrics           import CHRF
import warnings; warnings.filterwarnings('ignore')

nltk.download('wordnet', quiet=True)
nltk.download('punkt',   quiet=True)
nltk.download('omw-1.4', quiet=True)

def calc_bleu(ref, hyp):
    r, h = ref.lower().split(), hyp.lower().split()
    if not r or not h: return 0.0
    sm = SmoothingFunction().method1
    w  = {1:(1,0,0,0),2:(.5,.5,0,0),3:(.33,.33,.34,0)}.get(min(len(r),len(h),4),(.25,.25,.25,.25))
    return sentence_bleu([r], h, weights=w, smoothing_function=sm) * 100

def calc_rouge(ref, hyp):
    sc = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
    return {k: v.fmeasure*100 for k, v in sc.score(ref.lower(), hyp.lower()).items()}

def calc_meteor(ref, hyp):
    try: return meteor_score([ref.lower().split()], hyp.lower().split()) * 100
    except: return 0.0

def calc_chrf(ref, hyp): return CHRF().sentence_score(hyp, [ref]).score
def calc_exact_match(ref, hyp): return 100.0 if ref.lower().strip()==hyp.lower().strip() else 0.0
def calc_word_overlap(ref, hyp):
    r, h = set(ref.lower().split()), set(hyp.lower().split())
    return len(r&h)/max(len(r),1)*100 if r else 0.0

def calc_recall_at_k(ref_de, retrieved, k_values=[1,3,5,10,20]):
    ref_norm = normalize_tla(ref_de).lower()
    return {f'recall@{k}': 100.0 if any(
        fuzz.ratio(normalize_tla(r['translation']).lower(), ref_norm)>=70
        for r in retrieved[:k]) else 0.0 for k in k_values}

def calc_mrr(ref_de, retrieved):
    ref_norm = normalize_tla(ref_de).lower()
    for rank, r in enumerate(retrieved, 1):
        if fuzz.ratio(normalize_tla(r['translation']).lower(), ref_norm)>=70: return 100.0/rank
    return 0.0

def evaluate_pipeline(gardiner_input, reference_german='', reference_english='', include_english=True):
    result = full_pipeline(gardiner_input, verbose=False, include_english=include_english)
    hyp_de = result.get('german', ''); hyp_en = result.get('english', '')
    rouge_de = calc_rouge(reference_german, hyp_de) if reference_german else {}
    recall   = calc_recall_at_k(reference_german, result.get('rag_results',[])) if reference_german else {}
    mrr      = calc_mrr(reference_german, result.get('rag_results',[])) if reference_german else 0.0
    rouge_en = calc_rouge(reference_english, hyp_en) if (reference_english and hyp_en) else {}
    return {
        **result, 'reference_german': reference_german, 'reference_english': reference_english,
        'bleu_de'        : calc_bleu(reference_german, hyp_de)        if reference_german else 0.0,
        'rouge1_de'      : rouge_de.get('rouge1', 0.0),
        'rouge2_de'      : rouge_de.get('rouge2', 0.0),
        'rougeL_de'      : rouge_de.get('rougeL', 0.0),
        'meteor_de'      : calc_meteor(reference_german, hyp_de)      if reference_german else 0.0,
        'chrf_de'        : calc_chrf(reference_german, hyp_de)         if reference_german else 0.0,
        'exact_match_de' : calc_exact_match(reference_german, hyp_de) if reference_german else 0.0,
        'word_overlap_de': calc_word_overlap(reference_german, hyp_de) if reference_german else 0.0,
        'bleu_en'        : calc_bleu(reference_english, hyp_en)       if (reference_english and hyp_en) else 0.0,
        'rouge1_en'      : rouge_en.get('rouge1', 0.0),
        **recall, 'mrr': mrr, 'avg_retrieval_score': np.mean(list(recall.values())) if recall else 0.0,
    }

print('✅ Evaluation suite ready (v17)')

## CELL 14 — RAG vs Dict-LoRA vs LLM-Only Comparison

In [ ]:
def translate_llm_only(query_original):
    prompt = (
        'Du bist ein Experte für altägyptische Grammatik.\n\n'
        'Übersetze die folgende ältere ägyptische Transliteration ins Deutsche.\n'
        'Gib NUR die deutsche Übersetzung aus.\n\n'
        f'Ägyptische Transliteration:\n{query_original}\n\n'
        'Deutsche Übersetzung:\n'
    )
    inputs = seed_x_tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)
    inputs.pop('token_type_ids', None)
    dev = next(seed_x_model.parameters()).device; inputs = {k:v.to(dev) for k,v in inputs.items()}
    with torch.no_grad():
        out = seed_x_model.generate(**inputs, max_new_tokens=150, num_beams=1, do_sample=False,
            no_repeat_ngram_size=3, repetition_penalty=1.5,
            pad_token_id=seed_x_tokenizer.pad_token_id, eos_token_id=seed_x_tokenizer.eos_token_id)
    return seed_x_tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def translate_lora_only(assembled_words):
    lm = enrich_meanings_with_lora(assembled_words)
    ne = [v for v in lm.values() if v]
    return ', '.join(ne) if ne else ''

def run_full_comparison(num_samples=None):
    if df_test_global is None or len(df_test_global) == 0:
        print('❌ Test set not available.'); return None, None, None
    test_df = df_test_global if num_samples is None else df_test_global.head(num_samples)
    print('=' * 70)
    print(f'🆚 RAG vs Dict-LoRA vs LLM-ONLY ({len(test_df)} samples)')
    print('=' * 70)
    rag_list, llm_list, lora_list = [], [], []
    for idx in range(len(test_df)):
        row = test_df.iloc[idx]; q = row['transliteration']; ref_de = row['translation']
        rag = evaluate_pipeline(q, reference_german=ref_de); rag_list.append(rag)
        lg = translate_llm_only(q); rl = calc_rouge(ref_de, lg)
        llm_list.append({'bleu_de':calc_bleu(ref_de,lg),'rouge1_de':rl.get('rouge1',0.0),
                          'rouge2_de':rl.get('rouge2',0.0),'rougeL_de':rl.get('rougeL',0.0),
                          'meteor_de':calc_meteor(ref_de,lg),'chrf_de':calc_chrf(ref_de,lg),
                          'exact_match_de':calc_exact_match(ref_de,lg),'word_overlap_de':calc_word_overlap(ref_de,lg)})
        aw = assemble_words(q.strip().upper().split(), use_tla=True)
        ld = translate_lora_only(aw); rl2 = calc_rouge(ref_de, ld) if ld else {}
        lora_list.append({'bleu_de':calc_bleu(ref_de,ld) if ld else 0.0,
                           'rouge1_de':rl2.get('rouge1',0.0),'chrf_de':calc_chrf(ref_de,ld) if ld else 0.0})
        if (idx+1)%10==0: print(f'  {idx+1}/{len(test_df)} processed...')
    df_rag=pd.DataFrame(rag_list); df_llm=pd.DataFrame(llm_list); df_lora=pd.DataFrame(lora_list)
    metric_cols = ['bleu_de','rouge1_de','rouge2_de','rougeL_de','meteor_de','chrf_de','exact_match_de','word_overlap_de']
    print('\n📊 COMPARISON (German metrics)')
    print(f'{"Metric":<22} {"RAG+LoRA":>10} {"LoRA-only":>10} {"LLM-Only":>10}')
    print('-' * 57)
    for m in metric_cols:
        print(f'{m:<22} {df_rag[m].mean() if m in df_rag.columns else 0.0:>10.2f}'
              f' {df_lora[m].mean() if m in df_lora.columns else 0.0:>10.2f}'
              f' {df_llm[m].mean() if m in df_llm.columns else 0.0:>10.2f}')
    for name, df in [('rag',df_rag),('llm',df_llm),('lora',df_lora)]:
        df.to_csv(os.path.join(_WORK_ROOT, f'{name}_eval_v17.csv'), index=False)
    print('\n💾 Saved')
    return df_rag, df_llm, df_lora

# df_rag, df_llm, df_lora = run_full_comparison(num_samples=50)

## CELL 15 — K-Value Optimization

In [ ]:
def run_k_optimization(num_samples=10, k_min=5, k_max=205, k_step=5):
    global TOP_K_RAG
    if df_test_global is None or len(df_test_global)==0:
        print('❌ Test set not available.'); return None
    test_df = df_test_global.head(num_samples)
    samples = [{'q': r['transliteration'], 'ref_de': r['translation'],
                 'q_norm': normalize_tla(r['transliteration'])}
               for _, r in test_df.iterrows()]
    k_results = []
    for k in range(k_min, k_max, k_step):
        TOP_K_RAG = k; bleu_s, recall_s = [], []
        for s in samples:
            rag    = hybrid_search(s['q'], top_k=k)
            recall = calc_recall_at_k(s['ref_de'], rag, k_values=[k])
            german = _get_best_german_from_rag(rag) if rag else ''
            bleu_s.append(calc_bleu(s['ref_de'], german))
            recall_s.append(recall.get(f'recall@{k}', 0.0))
        ab = np.mean(bleu_s); ar = np.mean(recall_s); comp = 0.5*ab + 0.5*ar
        k_results.append({'k':k,'bleu_de':ab,f'recall@{k}':ar,'composite':comp})
        print(f'  K={k:3d} | BLEU(de)={ab:5.2f} | Recall={ar:5.2f} | Composite={comp:5.2f}')
    df_k = pd.DataFrame(k_results)
    br = df_k.loc[df_k['composite'].idxmax()]
    TOP_K_RAG = int(br['k'])
    print(f'\n🏆 Optimal K = {TOP_K_RAG} (composite = {br["composite"]:.2f})')
    df_k.to_csv(os.path.join(_WORK_ROOT, 'k_opt_v17.csv'), index=False)
    return df_k

# df_k = run_k_optimization(num_samples=10)

## CELL 16 — Kill Old Flask Process

In [ ]:
import os
os.system('kill -9 $(lsof -t -i:5010) 2>/dev/null')
time.sleep(2)
print('✅ Port 5010 cleared')

## CELL 17 — Flask API v17

In [ ]:
import threading
from pyngrok import ngrok, conf
from flask import Flask, request, jsonify
from flask_cors import CORS

ngrok.kill(); time.sleep(2)

NGROK_TOKEN = os.environ.get('NGROK_AUTH_TOKEN', '')
if not NGROK_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        NGROK_TOKEN = UserSecretsClient().get_secret('NGROK_AUTH_TOKEN')
    except Exception: pass
if not NGROK_TOKEN:
    print('⚠️  NGROK_AUTH_TOKEN not set')
conf.get_default().auth_token = NGROK_TOKEN

app = Flask(__name__); CORS(app)

ERROR_CODES = {
    'MISSING_INPUT':('400','Required field is missing'),
    'INVALID_INPUT':('400','Input is malformed or empty'),
    'PIPELINE_ERROR':('500','Internal pipeline failure'),
    'MODEL_ERROR':('503','Language model unavailable'),
}

def _api_error(code_key, detail='', status=None):
    code, msg = ERROR_CODES.get(code_key,('500','Unknown error'))
    return jsonify({'success':False,'error':{'code':code_key,'message':msg,'detail':detail}}), status or int(code)

def _run_pipeline_safe(gardiner_str, include_english=False):
    last_exc = None
    for _ in range(2):
        try: return full_pipeline(gardiner_str, verbose=False, include_english=include_english), None
        except RuntimeError as e:
            last_exc=e; gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        except Exception as e: return None, str(e)
    return None, str(last_exc)

@app.route('/api/decipher', methods=['POST'])
def decipher():
    data = request.get_json(silent=True)
    if not data: return _api_error('MISSING_INPUT','JSON body required')
    codes = data.get('codes',[])
    if not codes: return _api_error('MISSING_INPUT','"codes" list required')
    if not isinstance(codes,list) or not all(isinstance(c,str) for c in codes):
        return _api_error('INVALID_INPUT','"codes" must be list of strings')
    include_en   = bool(data.get('include_english',False))
    gardiner_str = ' '.join(codes).upper()
    result, err  = _run_pipeline_safe(gardiner_str, include_english=include_en)
    if err: return _api_error('PIPELINE_ERROR', err)
    code_list = result['codes_snapped']
    tla_toks  = result['tla_phonemes'].split()
    meanings  = result['meanings']
    per_sign  = [[c, GARDINER_MAP.get(c.strip().lower(),{}).get('unicode',''),
                  tla_toks[i] if i<len(tla_toks) else '',
                  meanings[i] if i<len(meanings) else '']
                 for i, c in enumerate(code_list)]
    glyphs = ''.join(GARDINER_MAP.get(c.strip().lower(),{}).get('unicode','') for c in code_list)
    intent = result.get('intention',{}); sent = result.get('sentiment',{}); rag_s = result['rag_score']
    return jsonify({'success':True,'data':{
        'path':result['path'],'glyphs':glyphs,'per_sign':per_sign,
        'snap_log':result.get('snap_log',[]),'phoneme_tokens':result.get('phoneme_tokens',[]),
        'tla_norm':result['tla_normalised'],
        'german':result['german'],'english':result.get('english',''),'arabic':result['arabic'],
        'arabic_verified':result.get('arabic_verified'),'arabic_confidence':result.get('arabic_confidence'),
        'sentiment':sent.get('label','neutral').lower(),'sent_score':f"{sent.get('confidence',0):.0%}",
        'intention_en':intent.get('intention_en',''),'intention_ar':intent.get('intention_ar',''),
        'intent_confidence':intent.get('confidence'),
        'rag_score':rag_s,'rag_score_max':RRF_SCORE_MAX,'rag_score_interpretation':interpret_rag_score(rag_s),
        'assembled_words':result.get('assembled_words',[]),
        'assembly_metadata':result.get('assembly_metadata',[]),
        'assembly_alternatives':result.get('assembly_alts',[]),
        'assembly_score':result.get('assembly_score'),
        'lora_meanings':result.get('lora_meanings',{}),'german_source':result.get('german_source',''),
        'time':result['processing_time'],
    }}), 200

@app.route('/api/translate', methods=['POST'])
def api_translate():
    data = request.get_json(silent=True)
    if not data: return _api_error('MISSING_INPUT','JSON body required')
    gardiner = data.get('gardiner','').strip()
    if not gardiner: return _api_error('MISSING_INPUT','"gardiner" required')
    result, err = _run_pipeline_safe(gardiner, include_english=bool(data.get('include_english',False)))
    if err: return _api_error('PIPELINE_ERROR', err)
    rag_s = result['rag_score']
    return jsonify({'success':True,
        'input':result['input'],'path':result['path'],
        'codes_snapped':result.get('codes_snapped',[]),'snap_log':result.get('snap_log',[]),
        'phoneme_tokens':result.get('phoneme_tokens',[]),
        'tla_phonemes':result['tla_phonemes'],'tla_normalised':result['tla_normalised'],
        'meanings':result['meanings'],
        'german':result['german'],'english':result.get('english',''),'arabic':result['arabic'],
        'arabic_verified':result.get('arabic_verified'),'sentiment':result.get('sentiment'),
        'intention':result.get('intention'),'lora_meanings':result.get('lora_meanings',{}),
        'rag_score':rag_s,'rag_score_max':RRF_SCORE_MAX,'rag_score_interpretation':interpret_rag_score(rag_s),
        'rag_results':result.get('rag_results',[])[:3],'german_source':result.get('german_source',''),
        'time':result['processing_time'],
    }), 200

@app.route('/api/evaluate', methods=['POST'])
def api_evaluate():
    data = request.get_json(silent=True)
    if not data: return _api_error('MISSING_INPUT','JSON body required')
    gardiner = data.get('gardiner','').strip()
    if not gardiner: return _api_error('MISSING_INPUT','"gardiner" required')
    ref_de = data.get('reference_german',''); ref_en = data.get('reference_english','')
    try:
        result = evaluate_pipeline(gardiner, reference_german=ref_de,
                                   reference_english=ref_en, include_english=bool(ref_en))
    except Exception as e: return _api_error('PIPELINE_ERROR', str(e))
    em = {k: result[k] for k in [
        'bleu_de','rouge1_de','rouge2_de','rougeL_de','meteor_de','chrf_de',
        'exact_match_de','word_overlap_de','bleu_en','rouge1_en',
        'recall@1','recall@3','recall@5','recall@10','recall@20','mrr','avg_retrieval_score',
    ] if k in result}
    return jsonify({'success':True,'input':gardiner,
                    'german':result['german'],'english':result.get('english',''),'metrics':em}), 200

@app.route('/api/status', methods=['GET'])
def status():
    return jsonify({
        'status':'ready','version':'v17-AccuracyOverhaul',
        'environment':ENV,'model':'Seed-X-PPO-7B',
        'primary_language':'German','secondary_language':'English (on request)',
        'dict_lora_ready':_LORA_READY,'lora_base_model':LORA_BASE_MODEL_ID,
        'signs_loaded':len(GARDINER_MAP),'valid_codes':len(VALID_CODES),
        'code_snapping':True,'royal_name_penalty':ROYAL_NAME_PENALTY,
        'faiss_vectors':faiss_index.ntotal if faiss_index else 0,
        'test_set_rows':len(df_test_global) if df_test_global is not None else 0,
        'intention_classes':len(INTENTION_MAP),
        'rag_k':RRF_K,'rag_alpha':HYBRID_ALPHA,'similarity_threshold':SIMILARITY_THRESHOLD,
    }), 200

@app.route('/api/health', methods=['GET'])
def health(): return jsonify({'status':'ok','version':'v17-AccuracyOverhaul'}), 200

@app.route('/api/examples', methods=['GET'])
def examples():
    return jsonify({
        'leiter_der_mitte' : {'codes':['S42','D2','F34'],'description':'Leiter der Mitte (Director of the heart-center)'},
        'horus_injury'     : {'codes':['S29','M17','G1','V13','D57','N35','S29','M17','D36','V13','D57','G5'],
                              'description':'Verletzung für den der Horus verletzt'},
        'snapped_codes'    : {'codes':['A01','G05','N35'],'description':'OCR-broken — snaps to A1 G5 N35'},
        'sun_disk'         : {'codes':['N5'],'description':'Sonne / Ra'},
        'royal_offering'   : {'codes':['R4','X8','A42'],'description':'Königsopfer'},
    }), 200

def run_server(): app.run(host='0.0.0.0', port=5010, use_reloader=False, debug=False)
threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)

tunnel     = ngrok.connect(5010)
PUBLIC_URL = tunnel.public_url

print('=' * 65)
print('🏛️  HIEROGLYPH NLP PIPELINE v17 — ACCURACY OVERHAUL')
print(f'🌐  {PUBLIC_URL}')
print('\n📡 Endpoints:')
print(f'      POST {PUBLIC_URL}/api/decipher')
print(f'      POST {PUBLIC_URL}/api/translate')
print(f'      POST {PUBLIC_URL}/api/evaluate')
print(f'      GET  {PUBLIC_URL}/api/status')
print('\n🔧 Fixes applied:')
print('   FIX 1: S42+D2+F34 → [ḫrp, ḥr, ib] (per-sign tokens, not fused)')
print('   FIX 2: A01→A1, G05→G5 (zero-stripping, not edit distance)')
print('   FIX 3: Royal name penalty — Senweseret no longer hijacks verb phrases')
print('   FIX 4: Semantic guard — proper nouns skipped for verb phrase queries')
print('   FIX 5: LoRA trained on German corpus pairs (not just English word hints)')
print('   FIX 6: Improved fallback: LoRA → sign meanings → LLM')
print('=' * 65)